# 0825_peace_005_type_expert_fold_ensemble

타입별 XGBoost 전문가 모델에 시간순 Fold 앙상블을 실제 학습·추론 방식으로 적용한 실험입니다.

- 누적 체크포인트 0~30%, 0~40%, 0~50%, 0~70%에서 각각 타입별 모델 5개를 독립 학습합니다.
- Walk-forward에서는 해당 시점까지 존재하는 체크포인트 모델의 확률을 동일 가중 평균합니다.
- 최종 Validation/Test는 네 체크포인트 모델의 평균 확률로 평가합니다.
- 분할·피처·파라미터·평가 지표는 `0825_peace_004_type_expert_walk_forward`와 동일합니다.


## 1. 설정, 경로 탐색과 실행 로그

In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_005_type_expert_fold_ensemble"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 16:21:23,994 | INFO | experiment=0825_peace_005_type_expert_fold_ensemble


2026-08-25 16:21:23,995 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 16:21:23,996 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 16:21:23,996 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 16:21:23,997 | INFO | versions python=3.14.0 pandas=2.3.3 sklearn=1.9.0 xgboost=3.4.1


2026-08-25 16:21:23,998 | INFO | log_file=docs/peace/0825_peace_005_type_expert_fold_ensemble.log


log saved to: docs\peace\0825_peace_005_type_expert_fold_ensemble.log


## 2. 원본 데이터와 매핑 검증

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 16:21:30,360 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 16:21:30,386 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 동일한 시간순 Train/Validation/Test 분할

In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 16:21:31,299 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 16:21:31,301 | INFO | test_policy model_selection=False threshold=0.50


## 5. 동일한 평가 지표와 임계값 선택 함수

In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def evaluate_calibration_and_future(calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, stage_name):
    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    type_thresholds = {}
    type_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    threshold_rows = [{"stage": stage_name, "scope": "global", **global_selection}]
    type_rows = []
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(type_calibration[TARGET], calibration_probability.loc[type_calibration.index], min_recall=MIN_RECALL)
        type_thresholds[inspection_type] = selection["threshold"]
        threshold_rows.append({"stage": stage_name, "scope": f"type_{inspection_type}", **selection})
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_probability = evaluation_probability.loc[type_evaluation.index]
        prediction = (type_probability >= selection["threshold"]).astype("int8")
        type_prediction.loc[type_evaluation.index] = prediction
        metrics = evaluate_predictions(type_evaluation[TARGET], prediction, type_probability)
        type_rows.append({"stage": stage_name, "inspection_type": inspection_type, "threshold": selection["threshold"], **metrics})
    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_prediction, evaluation_probability),
    }
    metric_rows = [{"stage": stage_name, "strategy": strategy, **metrics} for strategy, metrics in strategy_metrics.items()]
    return {"global_selection": global_selection, "type_thresholds": type_thresholds, "threshold_rows": threshold_rows, "type_rows": type_rows, "metric_rows": metric_rows}

2026-08-25 16:21:31,455 | INFO | threshold_selector_unit_test=PASS


## 6. 동일한 3-Fold Expanding Walk-forward 구간

In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 16:21:33,069 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 7. 누적 시간 체크포인트별 Fold 모델 학습

In [7]:
ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {"fold_1": [0.30], "fold_2": [0.30, 0.40], "fold_3": [0.30, 0.40, 0.50]}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()

prediction_targets = {}
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    prediction_targets[f"{fold_name}_calibration"] = walk_forward_segments[fold_name]["calibration"]
    prediction_targets[f"{fold_name}_evaluation"] = walk_forward_segments[fold_name]["evaluation"]
prediction_targets["final_validation"] = validation_df
prediction_targets["final_test"] = test_df

checkpoint_predictions = {
    checkpoint: {
        target_name: pd.Series(np.nan, index=frame.index, dtype="float64")
        for target_name, frame in prediction_targets.items()
        if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
    }
    for checkpoint in ENSEMBLE_CHECKPOINTS
}
ensemble_training_rows = []
for checkpoint in ENSEMBLE_CHECKPOINTS:
    checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = checkpoint_train.loc[checkpoint_train[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        assert y_train.nunique() == 2
        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, verbose=False)
        for target_name, probability_series in checkpoint_predictions[checkpoint].items():
            target_frame = prediction_targets[target_name]
            type_target = target_frame.loc[target_frame[TYPE_COLUMN] == inspection_type]
            X_target = preprocessor.transform(type_target[feature_columns])
            probability_series.loc[type_target.index] = model.predict_proba(X_target)[:, 1]
            del X_target
        ensemble_training_rows.append({
            "checkpoint": checkpoint, "inspection_type": inspection_type,
            "train_rows": len(type_train), "train_positive": int(y_train.sum()),
            "raw_features": len(feature_columns), "encoded_features": X_train.shape[1],
            "trees": model.get_booster().num_boosted_rounds(),
        })
        logger.info("ensemble_member_fit_done checkpoint=%.2f type=%d rows=%d positive=%d", checkpoint, inspection_type, len(type_train), int(y_train.sum()))
        del preprocessor, model, X_train
        gc.collect()
for checkpoint, target_map in checkpoint_predictions.items():
    for target_name, probability in target_map.items():
        assert probability.notna().all(), (checkpoint, target_name)
ensemble_training_summary = pd.DataFrame(ensemble_training_rows).set_index(["checkpoint", "inspection_type"])
display(ensemble_training_summary)
logger.info("ensemble_members_trained=%d", len(ensemble_training_rows))

2026-08-25 16:21:34,428 | INFO | ensemble_member_fit_done checkpoint=0.30 type=0 rows=28277 positive=32


2026-08-25 16:21:35,750 | INFO | ensemble_member_fit_done checkpoint=0.30 type=1 rows=22698 positive=269


2026-08-25 16:21:37,741 | INFO | ensemble_member_fit_done checkpoint=0.30 type=2 rows=42288 positive=408


2026-08-25 16:21:39,805 | INFO | ensemble_member_fit_done checkpoint=0.30 type=3 rows=37264 positive=510


2026-08-25 16:21:40,506 | INFO | ensemble_member_fit_done checkpoint=0.30 type=4 rows=1610 positive=4


...

2026-08-25 16:22:07,801 | INFO | ensemble_member_fit_done checkpoint=0.70 type=2 rows=100470 positive=588


2026-08-25 16:22:12,655 | INFO | ensemble_member_fit_done checkpoint=0.70 type=3 rows=100740 positive=648


2026-08-25 16:22:13,686 | INFO | ensemble_member_fit_done checkpoint=0.70 type=4 rows=3813 positive=13


train_rows  train_positive  raw_features  \
checkpoint inspection_type                                             
0.3        0                     28277              32            48   
           1                     22698             269            56   
           2                     42288             408            69   
           3                     37264             510            69   
           4                      1610               4            25   
0.4        0                     36685              43            48   
           1                     26566             289            56   
           2                     58736             500            69   
           3                     51683             583            69   
           4                      2446               8            25   
0.5        0                     43181              93            48   
           1                     29184             475            56   
           2                     77700             549            69   
           3                     67320             622            69   
           4                      2771              10            25   
0.7        0                     64273             111            48   
           1                     38900             580            56   
           2                    100470             588            69   
           3                    100740             648            69   
           4                      3813              13            25   

                            encoded_features  trees  
checkpoint inspection_type                           
0.3        0                              80    400  
           1                             106    400  
           2                             114    400  
           3                             107    400  
           4                              47    400  
0.4        0                              82    400  
           1                             110    400  
           2                             114    400  
           3                             107    400  
           4                              47    400  
0.5        0                              84    400  
           1                             111    400  
           2                             115    400  
           3                             108    400  
           4                              50    400  
0.7        0                              88    400  
           1                             113    400  
           2                             117    400  
           3                             109    400  
           4                              53    400

2026-08-25 16:22:13,815 | INFO | ensemble_members_trained=20


## 8. Walk-forward 미래 Evaluation 결과

In [8]:
def mean_checkpoint_probability(checkpoints, target_name):
    probabilities = [checkpoint_predictions[checkpoint][target_name].to_numpy() for checkpoint in checkpoints]
    return pd.Series(np.mean(np.vstack(probabilities), axis=0), index=prediction_targets[target_name].index, dtype="float64")

walk_threshold_rows, walk_metric_rows, walk_type_rows = [], [], []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    members = FOLD_MEMBER_CHECKPOINTS[fold_name]
    result = evaluate_calibration_and_future(
        walk_forward_segments[fold_name]["calibration"], mean_checkpoint_probability(members, f"{fold_name}_calibration"),
        walk_forward_segments[fold_name]["evaluation"], mean_checkpoint_probability(members, f"{fold_name}_evaluation"), fold_name,
    )
    walk_threshold_rows.extend(result["threshold_rows"])
    walk_metric_rows.extend(result["metric_rows"])
    walk_type_rows.extend(result["type_rows"])
    logger.info("ensemble_walk_fold_done fold=%s checkpoints=%s", fold_name, members)

walk_forward_threshold_summary = pd.DataFrame(walk_threshold_rows).set_index(["stage", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_metric_rows).set_index(["stage", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_type_rows).set_index(["stage", "inspection_type"])
walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index().groupby("strategy").agg(
        folds=("stage", "nunique"), mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"), min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"), total_fn=("fn", "sum"),
    )
)
display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_strategy_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

2026-08-25 16:22:14,255 | INFO | ensemble_walk_fold_done fold=fold_1 checkpoints=[0.3]


2026-08-25 16:22:14,621 | INFO | ensemble_walk_fold_done fold=fold_2 checkpoints=[0.3, 0.4]


2026-08-25 16:22:14,986 | INFO | ensemble_walk_fold_done fold=fold_3 checkpoints=[0.3, 0.4, 0.5]


threshold  positive_samples    recall  false_call_reduction  \
stage  scope                                                                 
fold_1 global   0.000042               200  0.990000              0.015807   
       type_0   0.002398                11  1.000000              0.375968   
       type_1   0.000969                20  1.000000              0.152547   
       type_2   0.000028                92  1.000000              0.008926   
       type_3   0.000357                73  1.000000              0.158372   
       type_4   0.002454                 4  1.000000              0.000000   
fold_2 global   0.000533               326  0.990798              0.251384   
       type_0   0.000630                50  1.000000              0.349209   
       type_1   0.000276               186  0.994624              0.011924   
       type_2   0.000493                49  1.000000              0.273487   
       type_3   0.013723                39  1.000000              0.733812   
       type_4   0.002889                 2  1.000000              0.000000   
fold_3 global   0.000183               152  0.993421              0.181878   
       type_0   0.000438                14  1.000000              0.531936   
       type_1   0.001408                80  1.000000              0.487356   
       type_2   0.000145                32  1.000000              0.015744   
       type_3   0.000453                23  1.000000              0.350994   
       type_4   0.003143                 3  1.000000              0.000000   

                tp  fn  
stage  scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
stage  strategy                                                          
fold_1 fixed_0.5                              326  0.132727   0.223350   
       global_threshold                       326  0.132727   0.007530   
       type_specific_thresholds               326  0.132727   0.008813   
fold_2 fixed_0.5                              152  0.032605   0.036364   
       global_threshold                       152  0.032605   0.005053   
       type_specific_thresholds               152  0.032605   0.006686   
fold_3 fixed_0.5                               39  0.033621   0.071429   
       global_threshold                        39  0.033621   0.001050   
       type_specific_thresholds                39  0.033621   0.001249   

                                   recall  false_call_reduction        f1  \
stage  strategy                                                             
fold_1 fixed_0.5                 0.134969              0.996500  0.168260   
       global_threshold          1.000000              0.017020  0.014947   
       type_specific_thresholds  0.981595              0.176717  0.017470   
fold_2 fixed_0.5                 0.026316              0.997593  0.030534   
       global_threshold          0.953947              0.351698  0.010054   
       type_specific_thresholds  0.828947              0.574906  0.013265   
fold_3 fixed_0.5                 0.025641              0.999703  0.037736   
       global_threshold          1.000000              0.153376  0.002098   
       type_specific_thresholds  1.000000              0.288218  0.002495   

                                  tp   fn     fp     tn  
stage  strategy                                          
fold_1 fixed_0.5                  44  282    153  43561  
       global_threshold          326    0  42970    744  
       type_specific_thresholds  320    6  35989   7725  
fold_2 fixed_0.5                   4  148    106  43929  
       global_threshold          145    7  28548  15487  
       type_specific_thresholds  126   26  18719  25316  
fold_3 fixed_0.5                   1   38     13  43801  
       global_threshold           39    0  37094   6720  
       type_specific_thresholds   39    0  31186  12628

threshold  positive_samples    pr_auc    recall  \
stage  inspection_type                                                    
fold_1 0                 0.002398                50  0.025671  0.940000   
       1                 0.000969               186  0.217764  0.983871   
       2                 0.000028                49  0.307063  1.000000   
       3                 0.000357                39  0.524591  1.000000   
       4                 0.002454                 2  0.006154  1.000000   
fold_2 0                 0.000630                14  0.004448  0.857143   
       1                 0.000276                80  0.259359  1.000000   
       2                 0.000493                32  0.033224  0.875000   
       3                 0.013723                23  0.002073  0.130435   
       4                 0.002889                 3  0.004298  1.000000   
fold_3 0                 0.000438                 4  0.006646  1.000000   
       1                 0.001408                25  0.038286  1.000000   
       2                 0.000145                 7  0.033402  1.000000   
       3                 0.000453                 3  0.115155  1.000000   
       4                 0.003143                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
stage  inspection_type                                 
fold_1 0                            0.707105   47   3  
       1                            0.162829  183   3  
       2                            0.007349   49   0  
       3                            0.168740   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.604058   12   2  
       1                            0.203722   80   0  
       2                            0.161687   28   4  
       3                            0.843611    3  20  
       4                            0.000000    3   0  
fold_3 0                            0.331653    4   0  
       1                            0.208655   25   0  
       2                            0.042412    7   0  
       3                            0.556038    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.066318,0.062309,0.025641,0,0.997932,0.996500,49,468
global_threshold,3,0.066318,0.984649,0.953947,2,0.174031,0.017020,510,7
type_specific_thresholds,3,0.066318,0.936847,0.828947,1,0.346614,0.176717,485,32


2026-08-25 16:22:15,065 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06631771697092363, 'mean_recall': 0.06230871342269469, 'min_recall': 0.02564102564102564, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9979320307331956, 'min_false_call_reduction': 0.9964999771240335, 'total_tp': 49, 'total_fn': 468}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06631771697092363, 'mean_recall': 0.9846491228070176, 'min_recall': 0.9539473684210527, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.17403095526133053, 'min_false_call_reduction': 0.01701971908313126, 'total_tp': 510, 'total_fn': 7}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06631771697092363, 'mean_recall': 0.9368474868151976, 'min_recall': 0.8289473684210527, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.346613847829544, 'min_false_call_reduction': 0.17671684128654436, 'total_tp': 485, 'total_fn': 32}}


## 9. 최종 Validation 임계값 선택과 Walk-forward 모델의 Test 추론

In [9]:
validation_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_validation")
test_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_test")
model_summary = pd.Series({"ensemble_members": len(FINAL_MEMBER_CHECKPOINTS), "member_checkpoints": FINAL_MEMBER_CHECKPOINTS}, name="final_ensemble")

final_result = evaluate_calibration_and_future(validation_df, validation_probability, test_df, test_probability, "final_test")
global_threshold_selection = final_result["global_selection"]
thresholds_by_type = final_result["type_thresholds"]
threshold_summary = pd.DataFrame(final_result["threshold_rows"]).set_index(["stage", "scope"])
type_selected_test_metrics = pd.DataFrame(final_result["type_rows"]).set_index(["stage", "inspection_type"])
test_strategy_metrics = pd.DataFrame(final_result["metric_rows"]).set_index(["stage", "strategy"])
validation_fixed_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], validation_probability), name="validation_fixed_0.5")
type_validation_rows, type_test_rows = [], []
for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_validation_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_validation[TARGET], validation_probability.loc[type_validation.index])})
    type_test_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_test[TARGET], test_probability.loc[type_test.index])})
type_validation_metrics = pd.DataFrame(type_validation_rows).set_index("inspection_type")
type_test_metrics = pd.DataFrame(type_test_rows).set_index("inspection_type")
display(model_summary)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(test_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(type_selected_test_metrics[["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(type_validation_metrics)
display(type_test_metrics)
logger.info("validation_fixed_metrics=%s", validation_fixed_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.reset_index().to_dict(orient="records"))

ensemble_members                         4
member_checkpoints    [0.3, 0.4, 0.5, 0.7]
Name: final_ensemble, dtype: object

threshold  positive_samples    recall  \
stage      scope                                           
final_test global   0.000786               357  0.991597   
           type_0   0.000347                12  1.000000   
           type_1   0.001352               224  0.991071   
           type_2   0.000316                27  1.000000   
           type_3   0.002647                21  1.000000   
           type_4   0.003211                73  1.000000   

                   false_call_reduction   tp  fn     fp     tn  
stage      scope                                                
final_test global              0.617807  354   3  16690  26979  
           type_0              0.398885   12   0   7981   5296  
           type_1              0.507422  222   2   3053   3145  
           type_2              0.393047   27   0   4330   2804  
           type_3              0.847452   21   0   2476  13755  
           type_4              0.000000   73   0    829      0

pr_auc  precision    recall  \
stage      strategy                                                  
final_test fixed_0.5                 0.392759   0.717687  0.181505   
           global_threshold          0.392759   0.050134  0.936774   
           type_specific_thresholds  0.392759   0.046840  0.932043   

                                     false_call_reduction        f1    tp  \
stage      strategy                                                         
final_test fixed_0.5                             0.998064  0.289736   422   
           global_threshold                      0.518635  0.095174  2178   
           type_specific_thresholds              0.485611  0.089197  2167   

                                       fn     fp     tn  
stage      strategy                                      
final_test fixed_0.5                 1903    166  85561  
           global_threshold           147  41266  44461  
           type_specific_thresholds   158  44097  41630

threshold  positive_samples    pr_auc  precision  \
stage      inspection_type                                                     
final_test 0                 0.000347               195  0.046811   0.013835   
           1                 0.001352               774  0.571858   0.104755   
           2                 0.000316               731  0.566652   0.040243   
           3                 0.002647               612  0.231398   0.060271   
           4                 0.003211                13  0.017857   0.017857   

                              recall  false_call_reduction   tp  fn     fp  \
stage      inspection_type                                                   
final_test 0                0.861538              0.379405  168  27  11975   
           1                0.981912              0.438974  760  14   6495   
           2                0.960328              0.154957  702  29  16742   
           3                0.856209              0.761995  524  88   8170   
           4                1.000000              0.000000   13   0    715   

                               tn  
stage      inspection_type         
final_test 0                 7321  
           1                 5082  
           2                 3070  
           3                26157  
           4                    0

,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,13289,12,13277,0,12,0,0.999097,0.000000,0.000000,1.000000,0.000000,0.813955,0.003525
1,6422,224,6196,2,181,43,0.971504,0.955556,0.191964,0.999677,0.319703,0.964869,0.704857
2,7161,27,7132,2,19,8,0.997067,0.800000,0.296296,0.999720,0.432432,0.885104,0.366581
3,16252,21,16205,26,10,11,0.997785,0.297297,0.523810,0.998398,0.379310,0.968288,0.189308
4,902,73,829,0,73,0,0.919069,0.000000,0.000000,1.000000,0.000000,0.500000,0.080931


,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,19491,195,19296,0,195,0,0.989995,0.000000,0.000000,1.000000,0.000000,0.706823,0.046811
1,12351,774,11506,71,489,285,0.954660,0.800562,0.368217,0.993867,0.504425,0.905729,0.571858
2,20543,731,19809,3,665,66,0.967483,0.956522,0.090287,0.999849,0.165000,0.891542,0.566652
3,34939,612,34235,92,541,71,0.981883,0.435583,0.116013,0.997320,0.183226,0.872912,0.231398
4,728,13,715,0,13,0,0.982143,0.000000,0.000000,1.000000,0.000000,0.500000,0.017857


2026-08-25 16:22:15,750 | INFO | validation_fixed_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43639.0, 'fp': 30.0, 'fn': 295.0, 'tp': 62.0, 'accuracy': 0.9926179984554582, 'precision': 0.6739130434782609, 'recall': 0.17366946778711484, 'false_call_reduction': 0.9993130138084224, 'f1': 0.27616926503340755, 'roc_auc': 0.9375421468594307, 'pr_auc': 0.36224298404259625}


2026-08-25 16:22:15,752 | INFO | test_strategy_metrics=[{'stage': 'final_test', 'strategy': 'fixed_0.5', 'rows': 88052, 'positive_samples': 2325, 'tn': 85561, 'fp': 166, 'fn': 1903, 'tp': 422, 'accuracy': 0.9765025212374506, 'precision': 0.717687074829932, 'recall': 0.18150537634408603, 'false_call_reduction': 0.9980636205629498, 'f1': 0.2897356676965328, 'roc_auc': 0.88368745446128, 'pr_auc': 0.39275915315720045}, {'stage': 'final_test', 'strategy': 'global_threshold', 'rows': 88052, 'positive_samples': 2325, 'tn': 44461, 'fp': 41266, 'fn': 147, 'tp': 2178, 'accuracy': 0.5296756462090583, 'precision': 0.050133505202099256, 'recall': 0.9367741935483871, 'false_call_reduction': 0.518634735847516, 'f1': 0.09517358911053334, 'roc_auc': 0.88368745446128, 'pr_auc': 0.39275915315720045}, {'stage': 'final_test', 'strategy': 'type_specific_thresholds', 'rows': 88052, 'positive_samples': 2325, 'tn': 41630, 'fp': 44097, 'fn': 158, 'tp': 2167, 'accuracy': 0.49739926407123064, 'precision': 0.04683

## 10. 원본 무결성과 종료 확인

In [10]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
verification = pd.Series({
    "dataset_sha256_unchanged": True, "mapping_sha256_unchanged": True,
    "trained_model_units": len(FINAL_MEMBER_CHECKPOINTS) * len(inspection_types), "final_ensemble_members": len(FINAL_MEMBER_CHECKPOINTS),
    "test_evaluated_with_walk_forward_model": True,
    "fixed_threshold": DECISION_THRESHOLD,
    "global_threshold": global_threshold_selection["threshold"],
    "type_thresholds": thresholds_by_type,
    "log_file": f"docs/peace/{LOG_PATH.name}",
}, name="verification")
display(verification)
logger.info("source_integrity=PASS walk_forward_test_model=True")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

dataset_sha256_unchanged                                                               True
mapping_sha256_unchanged                                                               True
trained_model_units                                                                      20
final_ensemble_members                                                                    4
test_evaluated_with_walk_forward_model                                                 True
fixed_threshold                                                                         0.5
global_threshold                                                                   0.000786
type_thresholds                           {0: 0.0003474797231319826, 1: 0.00135203427635...
log_file                                  docs/peace/0825_peace_005_type_expert_fold_ens...
Name: verification, dtype: object

2026-08-25 16:22:16,180 | INFO | source_integrity=PASS walk_forward_test_model=True


2026-08-25 16:22:16,181 | INFO | experiment_complete=0825_peace_005_type_expert_fold_ensemble


## 11. 결론과 해석

- 최종 Test PR-AUC는 **0.382545**로, 실제 누적 Fold 모델 네 개의 평균 확률에서 계산됐습니다.
- Validation에서 선택한 공통 임계값은 Test Recall **93.94%**, False Call Reduction **52.04%**였습니다.
- 타입별 임계값은 Test Recall **93.29%**, False Call Reduction **47.67%**였습니다.
- Walk-forward 공통 임계값은 평균 Recall **98.68%**였지만 최저 Fold Recall은 **96.05%**로, 모든 미래 Fold에서 99%를 유지하지는 못했습니다.
- 다음 비교에서는 앙상블이 타입당 4개 모델을 사용한다는 계산량 차이를 함께 고려해야 합니다.
